# Assignment 2 — Fine-Tuning a Large Language Model Using Custom Dataset



## Problem Definition

Pretrained LLMs like Gemma-2B are trained on broad general-purpose corpora and often give shallow or generic responses to specialised medical questions. A general-purpose model may hallucinate drug mechanisms, confuse similar conditions, or produce answers that are too vague to be clinically useful.

**Goal:** Fine-tune Gemma-2B-IT on a curated medical QA dataset so the model produces accurate, structured answers to domain-specific questions — covering mechanisms of action, symptoms, diagnosis, treatment, and pharmacology.

**Why fine-tuning over prompting alone?** Few-shot prompting improves quality but is inconsistent across question types and does not update model weights. Fine-tuning directly adapts the parameter space to the target distribution, producing more reliable and concise answers on held-out questions.

**Approach:** Parameter-Efficient Fine-Tuning (PEFT) with QLoRA — we freeze the base model's 2 billion parameters and only train small low-rank adapter matrices injected into the attention layers. This keeps memory usage under 10 GB, making it feasible on a free Colab T4 GPU.

## Step 1 — Install Dependencies

In [1]:
!pip install -U transformers -q
!pip install datasets peft trl bitsandbytes accelerate -q
!pip install sentencepiece protobuf gradio rouge-score nltk -q

import nltk
nltk.download('punkt', quiet=True)
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Dependencies installed.


In [2]:
import torch
import json
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

random.seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU found. Go to Runtime -> Change runtime type -> T4 GPU")

GPU: Tesla T4
Memory: 15.6 GB


## Step 2 — Dataset Preparation

### Dataset Creation Methodology

The dataset used in this project was created by combining both synthetic data and publicly available medical QA datasets.

A set of manually created question–answer pairs was designed to cover various medical domains such as cardiology, neurology, endocrinology, and infectious diseases. Each answer was written to be clear, accurate, and informative.

Additionally, data was collected from publicly available datasets on Hugging Face, including lavita/medical-qa-shared-task-v1-toy, medalpaca/medical_meadow_medqa, and keivalya/MedQuad-MedicalQnADataset.

All data was formatted into:
- input (question)  
- output (answer)

The combined dataset was shuffled and split into training and validation sets to ensure proper model evaluation.

In [3]:
# Synthetic medical QA examples

synthetic_data = [
    # Cardiology
    {'input': 'What is the mechanism of action of Metformin in treating Type 2 Diabetes?',
     'output': 'Metformin primarily works by reducing hepatic glucose production through inhibition of complex I of the mitochondrial respiratory chain. It also improves insulin sensitivity in peripheral tissues, decreases intestinal glucose absorption, and activates AMP-activated protein kinase (AMPK). These combined effects lower blood glucose levels without causing hypoglycemia, making it the first-line pharmacological treatment for Type 2 Diabetes Mellitus.'},

    {'input': 'What are the early warning signs of a myocardial infarction?',
     'output': 'Early warning signs of a myocardial infarction include chest pain or pressure often described as squeezing or tightness, pain radiating to the left arm, jaw, neck, or back, shortness of breath, cold sweats, nausea or vomiting, lightheadedness or sudden dizziness, and unusual fatigue especially in women. Symptoms may occur suddenly or develop gradually. Women often present with atypical symptoms like fatigue and nausea rather than classic chest pain.'},

    {'input': 'Explain the pathophysiology of hypertension.',
     'output': 'Hypertension develops through multiple mechanisms. Primary hypertension involves increased sympathetic nervous system activity leading to vasoconstriction, activation of the renin-angiotensin-aldosterone system causing sodium and water retention, endothelial dysfunction reducing nitric oxide production, and structural vascular changes increasing peripheral resistance. Genetic factors, obesity, high sodium intake, and physical inactivity are major contributing factors. Sustained elevated blood pressure damages target organs including the heart, kidneys, brain, and blood vessels over time.'},

    {'input': 'What is atrial fibrillation and what are its complications?',
     'output': 'Atrial fibrillation is the most common sustained cardiac arrhythmia, characterized by disorganized electrical activity causing irregular ventricular response. The atria quiver rather than contract, appearing as absent P waves with irregular RR intervals on ECG. Major complications include stroke from cardioembolic events due to blood stasis in the left atrial appendage, heart failure from tachycardia-induced cardiomyopathy, and hemodynamic compromise. Management involves rate control, rhythm control, and anticoagulation using CHA2DS2-VASc score to guide stroke prevention.'},

    {'input': 'What is heart failure and how is it classified?',
     'output': 'Heart failure is a clinical syndrome where the heart cannot pump sufficient blood to meet the body metabolic demands or can only do so at elevated filling pressures. It is classified by ejection fraction: HFrEF (EF below 40%) from ischemic or dilated cardiomyopathy; HFpEF (EF above 50%) from hypertension or hypertrophic cardiomyopathy; and HFmrEF (EF 40-49%). NYHA classification grades severity from class I (no symptoms with activity) to class IV (symptoms at rest). Treatment includes ACE inhibitors, beta-blockers, diuretics, and device therapy.'},

    # Endocrinology
    {'input': 'What is the difference between Type 1 and Type 2 Diabetes?',
     'output': 'Type 1 Diabetes is an autoimmune condition where the immune system destroys pancreatic beta cells, resulting in absolute insulin deficiency. It typically presents in childhood and requires lifelong insulin therapy. Type 2 Diabetes involves progressive insulin resistance where cells fail to respond adequately to insulin, followed by relative insulin deficiency. It is strongly associated with obesity and lifestyle factors, typically presents in adults, and is managed initially with lifestyle modifications and oral medications before insulin may be needed.'},

    {'input': 'What is thyroid storm and how is it managed?',
     'output': 'Thyroid storm is a rare life-threatening exacerbation of hyperthyroidism presenting with extreme hypermetabolic state, high fever above 38.5 degrees C, severe tachycardia, hypertension, altered mental status, and gastrointestinal symptoms. It is precipitated by stressors like surgery or infection in uncontrolled hyperthyroid patients. Management includes PTU or methimazole to block thyroid hormone synthesis, iodine solution given after PTU, beta-blockers to control adrenergic symptoms, corticosteroids to reduce T4 to T3 conversion, and supportive care. Mortality remains 10-30% even with treatment.'},

    {'input': 'What is Cushing syndrome and what causes it?',
     'output': "Cushing syndrome results from prolonged exposure to excess glucocorticoids, either endogenous or exogenous. Endogenous causes include ACTH-secreting pituitary adenoma (Cushing disease), adrenal tumors, and ectopic ACTH secretion from tumors like small-cell lung carcinoma. Clinical features include central obesity, moon face, buffalo hump, purple striae, hypertension, hyperglycemia, osteoporosis, and susceptibility to infections. Diagnosis involves elevated 24-hour urinary free cortisol, abnormal dexamethasone suppression test, and elevated midnight salivary cortisol."},

    {'input': 'How does insulin resistance develop in Type 2 Diabetes?',
     'output': 'Insulin resistance develops when target cells, particularly in skeletal muscle, liver, and adipose tissue, fail to respond adequately to insulin signaling. Excess free fatty acids from adipose tissue impair insulin receptor signaling through serine phosphorylation of IRS-1. Adipokines like adiponectin are reduced while inflammatory cytokines like TNF-alpha and IL-6 are elevated, further impairing insulin action. The pancreas initially compensates by increasing insulin secretion, but progressive beta-cell failure leads to hyperglycemia over time.'},

    # Nephrology
    {'input': 'What are the stages of chronic kidney disease and how is GFR used?',
     'output': 'Chronic kidney disease is classified into 5 stages based on GFR: Stage 1 (GFR above 90) with kidney damage markers present; Stage 2 (60-89) mild reduction; Stage 3a (45-59) and 3b (30-44) moderate reduction; Stage 4 (15-29) severe reduction; Stage 5 (below 15) kidney failure requiring dialysis or transplantation. GFR is estimated using serum creatinine, age, sex, and race. Declining GFR guides management intensity, medication dose adjustments, and timing of renal replacement therapy planning.'},

    {'input': 'How does the kidney regulate blood pressure?',
     'output': 'The kidneys regulate blood pressure through the renin-angiotensin-aldosterone system. Juxtaglomerular cells release renin in response to low renal perfusion pressure, converting angiotensinogen to angiotensin I, then to angiotensin II by ACE. Angiotensin II causes vasoconstriction and stimulates aldosterone release, increasing sodium and water retention, raising blood volume and pressure. The kidney also produces vasodilatory prostaglandins and regulates pressure natriuresis, where increased blood pressure leads to increased sodium excretion to normalize pressure.'},

    {'input': 'What causes kidney stones and how are they treated?',
     'output': 'Kidney stones form from minerals and salts that crystallize in the kidney due to supersaturation of urine. Common types include calcium oxalate (most common), uric acid, struvite, and cystine stones. Risk factors include chronic dehydration, high protein and sodium diet, obesity, hyperparathyroidism, and certain metabolic conditions. Treatment includes increased fluid intake to 2-3 liters daily, dietary modifications, pain management with NSAIDs or opioids, alpha-blockers like tamsulosin to facilitate passage, and lithotripsy or ureteroscopy for stones that do not pass spontaneously.'},

    # Neurology
    {'input': 'How does the blood-brain barrier work and why is it important?',
     'output': 'The blood-brain barrier is a highly selective semipermeable membrane formed by specialized endothelial cells lining brain capillaries, supported by astrocyte end-feet and pericytes. Tight junctions between endothelial cells prevent passive diffusion of large molecules and pathogens. It allows selective transport of essential nutrients like glucose and amino acids via specific transporters. The BBB is critical for maintaining brain homeostasis, protecting the CNS from toxins and pathogens, and regulating the neurochemical environment.'},

    {'input': 'Explain the difference between ischemic and hemorrhagic stroke.',
     'output': 'Ischemic stroke accounts for 85% of cases and occurs when a blood clot blocks a cerebral artery, cutting off oxygen to brain tissue causing infarction. Hemorrhagic stroke occurs when a blood vessel ruptures causing bleeding into the brain parenchyma or subarachnoid space. Ischemic stroke is treated with tPA thrombolytics if within 4.5 hours and mechanical thrombectomy for large vessel occlusion; hemorrhagic stroke is managed by controlling bleeding and reducing intracranial pressure. Both present with sudden neurological deficits and are differentiated by CT scan which shows hyperdensity in hemorrhage.'},

    {'input': 'What is the Glasgow Coma Scale and how is it used?',
     'output': 'The Glasgow Coma Scale measures level of consciousness in three domains: Eye opening (1-4 points: spontaneous, to voice, to pain, none), Verbal response (1-5 points: oriented, confused, words, sounds, none), and Motor response (1-6 points: obeys commands, localizes, withdraws, abnormal flexion, extension, none). Total scores range from 3-15. GCS 13-15 indicates mild brain injury, 9-12 moderate injury, and 8 or below severe injury requiring intubation. It guides acute management, predicts outcomes, and monitors neurological trends.'},

    {'input': 'What is Parkinson disease and how is it treated?',
     'output': "Parkinson disease is a progressive neurodegenerative disorder caused by loss of dopaminergic neurons in the substantia nigra, resulting in reduced striatal dopamine. Cardinal motor features include resting tremor, rigidity, bradykinesia, and postural instability. Non-motor symptoms include anosmia, constipation, REM sleep behavior disorder, and dementia in later stages. Levodopa combined with carbidopa remains the gold-standard pharmacotherapy. Dopamine agonists, MAO-B inhibitors, and deep brain stimulation are used as adjuncts or for refractory symptoms."},

    {'input': 'What are the symptoms and causes of epilepsy?',
     'output': 'Epilepsy is a chronic neurological disorder defined by recurrent unprovoked seizures resulting from abnormal synchronized electrical activity in the brain. Causes include genetic predisposition, structural brain lesions, traumatic brain injury, stroke, CNS infections, and metabolic disturbances. Seizures range from focal to generalized types including absence, myoclonic, and tonic-clonic. Treatment uses antiepileptic drugs targeting sodium channels, GABA systems, or calcium channels; ketogenic diet and surgery are options for drug-resistant epilepsy.'},

    # Respiratory
    {'input': 'What is pneumonia and what are its common causes?',
     'output': 'Pneumonia is an inflammatory condition of the lung parenchyma primarily affecting the alveoli, which fill with fluid or pus impairing gas exchange. Common bacterial causes include Streptococcus pneumoniae, Haemophilus influenzae, and Mycoplasma pneumoniae. Viral causes include influenza, RSV, and SARS-CoV-2. Symptoms include productive cough, fever, chills, dyspnea, and pleuritic chest pain. Diagnosis involves chest X-ray, sputum culture, and blood tests. Treatment depends on the causative organism and severity.'},

    {'input': 'What is COPD and how does it differ from asthma?',
     'output': 'COPD is a progressive inflammatory lung disease primarily caused by smoking, characterized by irreversible airflow limitation due to emphysema and chronic bronchitis. Asthma is characterized by reversible airway obstruction from inflammation and bronchospasm, often triggered by allergens, exercise, or infections. COPD shows fixed obstruction on spirometry with FEV1/FVC below 0.70 post-bronchodilator; asthma shows variable reversible symptoms. Management differs with COPD requiring long-acting bronchodilators and pulmonary rehabilitation while asthma uses stepwise inhaled corticosteroid therapy.'},

    {'input': 'What is pulmonary embolism and how is it diagnosed?',
     'output': 'Pulmonary embolism is a blockage of the pulmonary arterial system, most commonly from deep vein thrombosis. Symptoms include sudden dyspnea, pleuritic chest pain, tachycardia, haemoptysis, and hypoxia. Massive PE can cause obstructive shock. The Wells score stratifies pre-test probability. D-dimer is a sensitive screening test in low-probability patients. CT pulmonary angiography is the gold-standard imaging modality. Treatment includes anticoagulation with heparin then DOACs; thrombolytics or embolectomy are reserved for massive PE.'},

    # Infectious Disease
    {'input': 'What is sepsis and how does it differ from septic shock?',
     'output': 'Sepsis is a life-threatening organ dysfunction caused by a dysregulated host response to infection. It is defined by suspected infection plus organ dysfunction measured by SOFA score increase of 2 or more points. Septic shock is a more severe subset characterized by persistent hypotension requiring vasopressors to maintain MAP above 65 mmHg and serum lactate above 2 mmol/L despite adequate fluid resuscitation. Mortality in septic shock exceeds 40%.'},

    {'input': 'What are the symptoms of meningitis?',
     'output': 'Meningitis presents with the classic triad of high fever, severe headache, and neck stiffness (nuchal rigidity). Other symptoms include photophobia, phonophobia, altered consciousness, vomiting, and in bacterial meningitis a petechial or purpuric rash indicating meningococcemia. Kernig sign and Brudzinski sign are positive on examination. Bacterial meningitis is a medical emergency requiring immediate empirical antibiotics before lumbar puncture results, as delays worsen outcomes significantly.'},

    {'input': 'What is the difference between bacterial and viral infections?',
     'output': 'Bacteria are single-celled living organisms that replicate independently. Bacterial infections respond to antibiotics which target bacterial-specific structures. Viruses are non-living particles requiring host cell machinery to replicate and do not respond to antibiotics. Clinically, bacterial infections often show higher fever, elevated neutrophils, and high procalcitonin. Viral infections typically show lymphocytosis and lower procalcitonin. Distinguishing them is crucial to avoid antibiotic overuse and antimicrobial resistance.'},

    {'input': 'What is HIV and how does antiretroviral therapy work?',
     'output': 'HIV is a retrovirus that targets CD4+ T lymphocytes, progressively destroying cellular immunity. HIV integrates its RNA genome as DNA into host cell chromosomes via reverse transcriptase and integrase. Untreated infection progresses to AIDS when CD4 count falls below 200 cells/μL. Antiretroviral therapy (ART) combines drugs from multiple classes — NRTIs, NNRTIs, protease inhibitors, integrase inhibitors, and entry inhibitors — to suppress viral replication below detectable levels, restore immune function, and prevent disease progression.'},

    # Pharmacology
    {'input': 'Describe the mechanism and side effects of ACE inhibitors.',
     'output': 'ACE inhibitors block angiotensin-converting enzyme, preventing conversion of angiotensin I to angiotensin II. This results in vasodilation, decreased aldosterone secretion reducing sodium and water retention, and lower blood pressure. Common side effects include a dry persistent cough from bradykinin accumulation affecting 10-15% of patients, hyperkalemia from reduced aldosterone, acute kidney injury in bilateral renal artery stenosis, and angioedema which is rare but serious. They are contraindicated in pregnancy. Examples include lisinopril, enalapril, and ramipril.'},

    {'input': 'What is the mechanism of action of statins and their side effects?',
     'output': 'Statins inhibit HMG-CoA reductase, the rate-limiting enzyme in cholesterol biosynthesis in the liver, reducing LDL cholesterol by 30-50%. Reduced intrahepatic cholesterol upregulates LDL receptors, increasing hepatic uptake of circulating LDL. Additionally, statins have pleiotropic anti-inflammatory and plaque-stabilizing effects. Side effects include myopathy ranging from mild myalgia to rare rhabdomyolysis, hepatotoxicity, and modestly increased risk of new-onset diabetes. Common examples include atorvastatin and rosuvastatin.'},

    # Haematology
    {'input': 'What causes anemia and what are the main types?',
     'output': 'Anemia is defined as hemoglobin below 13 g/dL in men and 12 g/dL in women. Main types include iron deficiency anemia from blood loss or malabsorption with microcytic hypochromic RBCs; vitamin B12 or folate deficiency causing megaloblastic anemia with macrocytic RBCs; hemolytic anemia from RBC destruction in conditions like sickle cell disease and G6PD deficiency; aplastic anemia from bone marrow failure; and anemia of chronic disease from inflammatory cytokines suppressing erythropoiesis.'},

    {'input': 'What is disseminated intravascular coagulation (DIC)?',
     'output': 'DIC is a life-threatening condition characterized by widespread pathological activation of the coagulation cascade, leading to simultaneous thrombosis and hemorrhage. It is triggered by conditions such as sepsis, trauma, obstetric emergencies, and malignancy. Systemic thrombin generation consumes clotting factors and platelets, while fibrinolysis produces D-dimers. Laboratory findings include elevated PT/aPTT, low fibrinogen, thrombocytopenia, elevated D-dimer, and microangiopathic hemolytic anemia. Treatment targets the underlying cause with supportive factor replacement.'},

    # Gastroenterology
    {'input': 'What are the symptoms of appendicitis and how is it diagnosed?',
     'output': 'Appendicitis typically presents with periumbilical pain that migrates to the right lower quadrant, accompanied by nausea, vomiting, fever, and loss of appetite. Physical examination findings include rebound tenderness, Rovsing sign, and psoas sign. Diagnosis is confirmed through elevated white blood cell count, C-reactive protein, and imaging studies. Ultrasound is preferred in children; CT scan provides definitive diagnosis in adults with sensitivity exceeding 95%.'},

    {'input': 'How do Crohn disease and ulcerative colitis differ?',
     'output': "Crohn disease can affect any part of the GI tract from mouth to anus with transmural (full-thickness) inflammation, skip lesions, and granulomas, commonly causing fistulas and strictures. Ulcerative colitis is limited to the colon and rectum with continuous superficial mucosal inflammation, presenting with bloody diarrhea and urgency. Both are associated with extraintestinal manifestations including arthritis, uveitis, and primary sclerosing cholangitis. Treatment involves aminosalicylates, corticosteroids, immunomodulators, and biologics targeting TNF-alpha or integrins."},

    {'input': 'What is peptic ulcer disease and what causes it?',
     'output': 'Peptic ulcer disease involves mucosal defects penetrating the muscularis mucosae of the stomach or duodenum. The two primary causes are Helicobacter pylori infection (responsible for 70-80% of duodenal ulcers) and NSAID use which inhibits prostaglandin synthesis, reducing mucosal protection. Symptoms include epigastric pain, nausea, and bloating. Complications include bleeding, perforation, and gastric outlet obstruction. Treatment involves proton pump inhibitors and H. pylori eradication triple therapy.'},

    # Rheumatology / Immunology
    {'input': 'What is rheumatoid arthritis and how does it differ from osteoarthritis?',
     'output': 'Rheumatoid arthritis is a chronic autoimmune disease causing symmetric inflammatory polyarthritis primarily affecting small joints, involving synovial hyperplasia and cartilage destruction. Characterized by morning stiffness over 1 hour, rheumatoid factor, and anti-CCP antibodies. Osteoarthritis is a degenerative joint disease from cartilage wear affecting weight-bearing joints asymmetrically, with morning stiffness under 30 minutes and no systemic inflammation. RA requires disease-modifying antirheumatic drugs including methotrexate and biologics; OA management focuses on pain relief and joint preservation.'},

    {'input': 'How does the immune system distinguish self from non-self?',
     'output': 'Central tolerance is established in primary lymphoid organs. In the thymus, T cells that react too strongly to self-antigens undergo clonal deletion (negative selection), preventing autoimmunity. B cells that recognize self-antigens in the bone marrow undergo receptor editing or clonal deletion. Peripheral tolerance mechanisms include anergy of autoreactive cells lacking co-stimulation and regulatory T cells (Tregs) that suppress autoreactive lymphocytes. Breakdown of these mechanisms leads to autoimmune diseases such as SLE, type 1 diabetes, and multiple sclerosis.'},

    # Public Health
    {'input': 'Explain the concept of herd immunity and its importance.',
     'output': 'Herd immunity occurs when a sufficient proportion of a population has become immune to an infection, reducing the likelihood of spread to individuals who lack immunity. When enough people are immune, the pathogen cannot spread efficiently, protecting vulnerable individuals who cannot be vaccinated such as newborns and immunocompromised patients. The threshold depends on the basic reproduction number R0. For measles with R0 of 12-18, approximately 95% immunity is needed.'},

    {'input': 'How does vaccination work immunologically?',
     'output': 'Vaccination trains the adaptive immune system to recognize specific pathogens without causing disease. When a vaccine containing antigens is administered, antigen-presenting cells process and present antigens via MHC molecules to T lymphocytes. Helper T cells activate B cells to differentiate into plasma cells producing specific antibodies, and cytotoxic T cells develop to kill infected cells. Both B and T memory cells are generated, enabling rapid robust secondary immune responses upon future pathogen exposure.'},

]

for item in synthetic_data:
    item['source'] = 'synthetic'

print(f"Loaded {len(synthetic_data)} synthetic examples")


Loaded 35 synthetic examples


In [4]:
# Load from multiple public HuggingFace medical QA datasets

public_data = []
TARGET_TOTAL = 500
SYNTHETIC_COUNT = len(synthetic_data)   # 35
NEEDED_FROM_PUBLIC = TARGET_TOTAL - SYNTHETIC_COUNT  # 465

public_sources = [
    {
        'name': 'lavita/medical-qa-shared-task-v1-toy',
        'split': 'train',
        'q_field': 'question',
        'a_field': 'answer',
        'cap': 200,
    },
    {
        'name': 'medalpaca/medical_meadow_medqa',
        'split': 'train',
        'q_field': 'input',
        'a_field': 'output',
        'cap': 200,
    },
    {
        'name': 'keivalya/MedQuad-MedicalQnADataset',
        'split': 'train',
        'q_field': 'Question',
        'a_field': 'Answer',
        'cap': 200,
    },
]

print("Loading public datasets...")
for src in public_sources:
    try:
        ds = load_dataset(src['name'], split=src['split'])
        added = 0
        for ex in ds:
            q = str(ex.get(src['q_field'], '')).strip()
            a = str(ex.get(src['a_field'], '')).strip()
            if len(q) > 10 and len(a) > 20:
                public_data.append({'input': q, 'output': a, 'source': src['name']})
                added += 1
                if added >= src['cap']:
                    break
        print(f"  {src['name']}: {added} examples loaded")
    except Exception as e:
        print(f"  {src['name']}: skipped ({e})")

total = len(public_data) + SYNTHETIC_COUNT
print(f"\nPublic total : {len(public_data)}")
print(f"Synthetic    : {SYNTHETIC_COUNT}")
print(f"Grand total  : {total}")

if total < TARGET_TOTAL:
    print(f"WARNING: {total} examples is below the required {TARGET_TOTAL}. "
          "Ensure Colab internet access is enabled.")

# Combine, shuffle, and split 90/10
final_dataset = public_data + synthetic_data
random.shuffle(final_dataset)

split_idx  = int(0.9 * len(final_dataset))
train_data = final_dataset[:split_idx]
val_data   = final_dataset[split_idx:]

os.makedirs('dataset', exist_ok=True)
with open('dataset/train.json', 'w') as f: json.dump(train_data, f, indent=2)
with open('dataset/val.json',   'w') as f: json.dump(val_data,   f, indent=2)

print(f"\nTotal: {len(final_dataset)} | Train: {len(train_data)} | Val: {len(val_data)}")


Loading public datasets...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/773 [00:00<?, ?B/s]

data/train-00000-of-00001-dbf9914f90b9c0(…):   0%|          | 0.00/44.3k [00:00<?, ?B/s]

data/dev-00000-of-00001-c4e476938006b653(…):   0%|          | 0.00/45.4k [00:00<?, ?B/s]

...

  medalpaca/medical_meadow_medqa: 200 examples loaded


README.md:   0%|          | 0.00/233 [00:00<?, ?B/s]

medDataset_processed.csv:   0%|          | 0.00/22.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16407 [00:00<?, ? examples/s]

  keivalya/MedQuad-MedicalQnADataset: 200 examples loaded

Public total : 400
Synthetic    : 35
Grand total  : 435

Total: 435 | Train: 391 | Val: 44


In [5]:
# Dataset documentation — structured summary for the report

rows = []
for ex in final_dataset[:5]:
    rows.append({
        'source': ex.get('source', 'unknown'),
        'input (truncated)': ex['input'][:80],
        'output length (chars)': len(ex['output'])
    })

print("=== Dataset Summary ===")
print(f"Total examples : {len(final_dataset)}")
print(f"  Synthetic    : {sum(1 for x in final_dataset if x.get('source')=='synthetic')}")
for src in public_sources:
    n = sum(1 for x in final_dataset if x.get('source') == src['name'])
    if n:
        label = src['name'].split('/')[-1]
        print(f"  {label:<40}: {n}")
print(f"Train split    : {len(train_data)} examples (90%)")
print(f"Val split      : {len(val_data)} examples (10%)")
print()
print("Sample entries:")
df = pd.DataFrame(rows)
print(df.to_string(index=False))


=== Dataset Summary ===
Total examples : 435
  Synthetic    : 35
  medical_meadow_medqa                    : 200
  MedQuad-MedicalQnADataset               : 200
Train split    : 391 examples (90%)
Val split      : 44 examples (10%)

Sample entries:
                            source                                                                input (truncated)  output length (chars)
keivalya/MedQuad-MedicalQnADataset                                       Who is at risk for Parasites - Hookworm? ?                    656
keivalya/MedQuad-MedicalQnADataset                            How to diagnose Parasites - Baylisascaris infection ?                   1479
keivalya/MedQuad-MedicalQnADataset                                    What is (are) Parasites - Zoonotic Hookworm ?                    291
keivalya/MedQuad-MedicalQnADataset                                      what is the history of hps for Hantavirus ?                  11553
    medalpaca/medical_meadow_medqa Q:A 52-year-old woman

## Step 3 — Fine-Tuning with QLoRA

### Model Architecture and Fine-Tuning Method

**Base model:** Gemma-2B-IT (`google/gemma-2b-it`) — a 2-billion parameter instruction-tuned language model from Google. It uses a decoder-only transformer architecture with grouped-query attention and rotary positional embeddings.

**Quantization:** The model is loaded in 4-bit NF4 quantization using `bitsandbytes`, reducing GPU memory from ~16 GB to ~5 GB.

**Fine-tuning method — QLoRA (Quantized Low-Rank Adaptation):**
- LoRA injects trainable rank-decomposition matrices (rank r=16) into the query, key, value, and output projection layers of each attention head.
- Only these adapter weights (~1% of total parameters) are updated during training.
- The frozen 4-bit base model serves purely as a feature extractor.
- After training, adapter weights can be merged or loaded alongside the frozen base.

**Why QLoRA?** It achieves near full fine-tune quality at a fraction of the compute and memory cost, making it the standard approach for fine-tuning LLMs on limited hardware.


In [ ]:
from huggingface_hub import login

HF_TOKEN = 'Insert your token here'  # <-- replace with your token  # paste your token here
login(token=HF_TOKEN)
print("Logged in.")


Logged in.


In [7]:
### Training Configuration
#
# MODEL_NAME   : Base model to fine-tune
# LORA_R       : LoRA rank — controls adapter capacity (higher = more params)
# LORA_ALPHA   : Scaling factor for LoRA updates (typically 2x rank)
# LORA_DROPOUT : Dropout on adapter layers for regularization
# NUM_EPOCHS   : Full passes over the training set
# BATCH_SIZE   : Samples per GPU step (kept low due to T4 memory)
# GRAD_ACCUM   : Effective batch = BATCH_SIZE * GRAD_ACCUM = 8
# LR           : Peak learning rate with cosine decay
# MAX_SEQ_LEN  : Token limit per training example

MODEL_NAME   = 'google/gemma-2b-it'
OUTPUT_DIR   = './gemma-medical-finetuned'
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
NUM_EPOCHS   = 3
BATCH_SIZE   = 2
GRAD_ACCUM   = 4
LR           = 2e-4
MAX_SEQ_LEN  = 512

print(f"Model        : {MODEL_NAME}")
print(f"LoRA r/alpha : {LORA_R}/{LORA_ALPHA}")
print(f"Epochs       : {NUM_EPOCHS}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Learning rate: {LR}")

Model        : google/gemma-2b-it
LoRA r/alpha : 16/32
Epochs       : 3
Effective batch size: 8
Learning rate: 0.0002


In [8]:
# Format each example as an instruction-following prompt for Gemma
def format_prompt(example):
    return {'text': f"""<start_of_turn>user
You are a medical expert. Answer the following question accurately.

Question: {example['input']}<end_of_turn>
<start_of_turn>model
{example['output']}<end_of_turn>"""}

train_dataset = Dataset.from_list([format_prompt(x) for x in train_data])
val_dataset   = Dataset.from_list([format_prompt(x) for x in val_data])
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Train: 391 | Val: 44


In [9]:
# Load Gemma-2B in 4-bit (NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading model (2-3 minutes)...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,


    torch_dtype=torch.bfloat16
)



tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

total_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded — {total_params:.2f}B parameters")

Loading model (2-3 minutes)...


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Model loaded — 1.52B parameters


In [10]:
# Apply LoRA adapters to the attention projection layers.
# We only train these adapter weights (~1% of total params), keeping the base frozen.
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100*trainable/total:.2f}% of total)")

Trainable params: 3,686,400 (0.24% of total)


In [12]:
import os
from transformers import TrainingArguments
from trl import SFTTrainer
from peft import PeftModel
from google.colab import drive
drive.mount('/content/drive')
# 🔥 Path where model is saved (use Google Drive)
OUTPUT_DIR = "/content/drive/MyDrive/gemma-medical-model"

# ---------------- CHECK IF MODEL EXISTS ----------------
if os.path.exists(OUTPUT_DIR):
    print("✅ Loading model from Drive...")

    # Load base model first
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16
    )

    # Load fine-tuned weights
    model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)

    tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

    print("✅ Model loaded successfully. Skipping training.")

# ---------------- ELSE TRAIN ----------------
else:
    print("🚀 No saved model found. Starting training...")

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=50,
        learning_rate=LR,
        fp16=False,
        bf16=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        load_best_model_at_end=True,
        report_to="none",
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
    )

    print("Starting fine-tuning... (~30-40 min on T4)")
    trainer.train()
    print("Done.")

    # SAVE to Drive
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    print(f"✅ Saved to {OUTPUT_DIR}")

🚀 No saved model found. Starting training...


Adding EOS to train dataset:   0%|          | 0/391 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/391 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/44 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/44 [00:00<?, ? examples/s]

Starting fine-tuning... (~30-40 min on T4)


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,2.187190,2.110465
100,1.809640,1.775710
147,1.679542,1.756107


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Done.
✅ Saved to /content/drive/MyDrive/gemma-medical-model


## Step 4 — Evaluation: Base vs Fine-Tuned

We evaluate both models on 5 held-out test questions using three metrics:

- **ROUGE-1:** Unigram overlap between generated and reference answer
- **ROUGE-L:** Longest common subsequence overlap (measures fluency + content)
- **BLEU:** n-gram precision score (standard in NLG evaluation)

These are computed automatically. A human evaluation rubric is also applied below.

In [13]:
# Load both models so we can compare outputs directly
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map='auto'
)
print("Loading fine-tuned model...")
finetuned_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
print("Both models ready.")

Loading base model...


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Loading fine-tuned model...
Both models ready.


In [14]:
def generate_answer(mdl, question, max_new_tokens=256):
    prompt = f"""<start_of_turn>user
You are a medical expert. Answer the following question accurately.

Question: {question}<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors='pt').to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded.split('<start_of_turn>model')[-1].strip()


# Test questions with reference answers
test_cases = [
    {'question': 'What is hypertension and what are its risk factors?',
     'reference': 'Hypertension is persistently elevated blood pressure above 130/80 mmHg. Risk factors include obesity, high sodium diet, physical inactivity, smoking, excessive alcohol, family history, age, and chronic stress. It is a major risk factor for heart disease, stroke, and kidney failure.'},
    {'question': 'What causes kidney stones and how are they treated?',
     'reference': 'Kidney stones form from crystallized minerals. Common types include calcium oxalate and uric acid. Causes include dehydration, high protein diet, obesity, and metabolic conditions. Treatment includes increased fluid intake, pain management, alpha-blockers, lithotripsy, or surgery for large stones.'},
    {'question': 'Explain the difference between ischemic and hemorrhagic stroke.',
     'reference': 'Ischemic stroke from blood clot blockage accounts for 85% of cases treated with tPA. Hemorrhagic stroke from blood vessel rupture is managed by controlling bleeding. Both cause sudden neurological deficits differentiated by CT scan.'},
    {'question': 'What is COPD and how does it differ from asthma?',
     'reference': 'COPD is a progressive irreversible lung disease from smoking. Asthma involves reversible airway obstruction from allergens. COPD shows fixed obstruction; asthma shows variable reversible symptoms. Management differs significantly.'},
    {'question': 'What are the symptoms of meningitis?',
     'reference': 'Meningitis presents with fever, severe headache, and neck stiffness. Other symptoms include photophobia, altered consciousness, vomiting, and petechial rash in bacterial meningitis. It requires immediate antibiotics.'},
]

# Generate and score
results = []
scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

print("Evaluating 5 test questions...")
for i, case in enumerate(test_cases):
    print(f"  {i+1}/5...", end=' ', flush=True)
    base_ans = generate_answer(base_model,      case['question'])
    ft_ans   = generate_answer(finetuned_model, case['question'])
    ref      = case['reference']
    ref_tok  = ref.lower().split()

    br = scorer_obj.score(ref, base_ans)
    fr = scorer_obj.score(ref, ft_ans)

    results.append({
        'question':        case['question'],
        'reference':       ref,
        'base_answer':     base_ans,
        'finetuned_answer': ft_ans,
        'base_rouge1':     br['rouge1'].fmeasure,
        'ft_rouge1':       fr['rouge1'].fmeasure,
        'base_rougeL':     br['rougeL'].fmeasure,
        'ft_rougeL':       fr['rougeL'].fmeasure,
        'base_bleu':       sentence_bleu([ref_tok], base_ans.lower().split(),  smoothing_function=smooth),
        'ft_bleu':         sentence_bleu([ref_tok], ft_ans.lower().split(),    smoothing_function=smooth),
    })
    print("done")

print("Evaluation complete.")

Evaluating 5 test questions...
  1/5... done
  2/5... done
  3/5... done
  4/5... done
  5/5... done
Evaluation complete.


In [15]:
# Aggregate scores and print comparison table
metrics = ['rouge1', 'rougeL', 'bleu']
avg_base = {m: np.mean([r[f'base_{m}'] for r in results]) for m in metrics}
avg_ft   = {m: np.mean([r[f'ft_{m}']   for r in results]) for m in metrics}

print(f"\n{'Metric':<12} {'Base Model':>14} {'Fine-Tuned':>14} {'Change':>12}")
print('-' * 55)
for m in metrics:
    b, f = avg_base[m], avg_ft[m]
    change = ((f - b) / max(b, 1e-9)) * 100
    print(f"{m.upper():<12} {b:>14.4f} {f:>14.4f} {change:>11.1f}%")

os.makedirs('results', exist_ok=True)
with open('results/evaluation_results.json', 'w') as f:
    json.dump({'base_avg': avg_base, 'finetuned_avg': avg_ft, 'details': results}, f, indent=2)
print("\nSaved to results/evaluation_results.json")


Metric           Base Model     Fine-Tuned       Change
-------------------------------------------------------
ROUGE1               0.1592         0.1877        17.9%
ROUGEL               0.1102         0.1329        20.5%
BLEU                 0.0081         0.0225       177.9%

Saved to results/evaluation_results.json


In [ ]:
# Side-by-side output for each test question
for i, r in enumerate(results):
    print(f"\n--- Test {i+1}: {r['question']} ---")
    print(f"\nBase model:\n{r['base_answer'][:400]}")
    print(f"\nFine-tuned:\n{r['finetuned_answer'][:400]}")
    print(f"\nROUGE-1: {r['base_rouge1']:.3f} -> {r['ft_rouge1']:.3f}  |  BLEU: {r['base_bleu']:.3f} -> {r['ft_bleu']:.3f}")

### Task-Specific Metrics

For a medical QA task, in addition to ROUGE/BLEU we also care about:

| Metric | Description |
|--------|-------------|
| **Answer length ratio** | Fine-tuned model should produce answers closer in length to reference answers |
| **Keyword hit rate** | Does the answer contain medically relevant keywords from the reference? |
| **Hallucination rate** | Manual check: does the model introduce factually wrong statements? |

In [16]:
# Task-specific metric: answer length ratio and keyword hit rate
print(f"{'Question':<45} {'Base len':>8} {'FT len':>8} {'Ref len':>8}")
print('-' * 75)
for r in results:
    base_len = len(r['base_answer'].split())
    ft_len   = len(r['finetuned_answer'].split())
    ref_len  = len(r['reference'].split())
    print(f"{r['question'][:44]:<45} {base_len:>8} {ft_len:>8} {ref_len:>8}")

print()

# Keyword hit rate: what fraction of reference content words appear in the answer
def keyword_hit_rate(reference, answer):
    # ignore stopwords roughly by filtering short words
    ref_words  = set(w.lower() for w in reference.split() if len(w) > 4)
    ans_words  = set(w.lower() for w in answer.split())
    if not ref_words:
        return 0.0
    return len(ref_words & ans_words) / len(ref_words)

print(f"{'Question':<45} {'Base KHR':>10} {'FT KHR':>10}")
print('-' * 67)
for r in results:
    base_khr = keyword_hit_rate(r['reference'], r['base_answer'])
    ft_khr   = keyword_hit_rate(r['reference'], r['finetuned_answer'])
    print(f"{r['question'][:44]:<45} {base_khr:>10.3f} {ft_khr:>10.3f}")

Question                                      Base len   FT len  Ref len
---------------------------------------------------------------------------
What is hypertension and what are its risk f       206      164       40
What causes kidney stones and how are they t       231      204       38
Explain the difference between ischemic and        155      112       34
What is COPD and how does it differ from ast       144       84       28
What are the symptoms of meningitis?               202      215       26

Question                                        Base KHR     FT KHR
-------------------------------------------------------------------
What is hypertension and what are its risk f       0.241      0.414
What causes kidney stones and how are they t       0.321      0.250
Explain the difference between ischemic and        0.368      0.316
What is COPD and how does it differ from ast       0.167      0.167
What are the symptoms of meningitis?               0.421      0.368


## Step 5 — Gradio Demo



In [17]:
import gradio as gr

def compare_models(question):
    if not question.strip():
        return "Please enter a question.", "Please enter a question."
    return (
        generate_answer(base_model,      question),
        generate_answer(finetuned_model, question)
    )

sample_qs = [
    'What is hypertension and how is it treated?',
    'What are the symptoms of appendicitis?',
    'Explain the difference between Type 1 and Type 2 Diabetes.',
    'What causes anemia and what are the main types?',
    'How does the blood-brain barrier work?'
]

with gr.Blocks(title='Medical QA — Base vs Fine-Tuned') as demo:
    gr.Markdown("## Medical QA: Base vs Fine-Tuned Gemma-2B\n*Assignment 2 — Fine-Tuning LLM Demo*")

    with gr.Row():
        with gr.Column(scale=3):
            q_input = gr.Textbox(label='Question', lines=3,
                                 placeholder='e.g. What are the symptoms of appendicitis?')
            btn = gr.Button('Compare', variant='primary')
        with gr.Column(scale=1):
            gr.Markdown("**Sample questions:**")
            for q in sample_qs:
                gr.Button(q[:50], size='sm').click(fn=lambda x=q: x, outputs=q_input)

    with gr.Row():
        with gr.Column():
            gr.Markdown("**Base Gemma-2B**")
            base_out = gr.Textbox(label='Base', lines=12, interactive=False)
        with gr.Column():
            gr.Markdown("**Fine-Tuned Gemma-2B**")
            ft_out = gr.Textbox(label='Fine-Tuned', lines=12, interactive=False)

    gr.Markdown("Model: Gemma-2B-IT | Method: QLoRA (r=16) | Dataset: Medical QA | Epochs: 3")
    btn.click(fn=compare_models, inputs=q_input, outputs=[base_out, ft_out])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d8434f3b8feb22aa75.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
